# Chapter 1 — Python for Production
### ML Engineer (Production) Interview Playbook

**Topics:** OOP · Typing · Generators · Decorators · Context Managers · Async · Complexity · Clean Code

> In the ML Engineer (Production) role, Python isn't just a tool for writing models it's the primary
> language of services that must be stable, testable, fast, and maintainable. The difference between a
> notebook script and production code lies in these exact details: resource management, concurrency,
> readability, and predictability of behavior. A technical interviewer usually asks not just *"does it
> work?"* but *"why did you make this decision, and what was the trade-off?"* This chapter focuses on
> that **"why."**
>
> **Interview tip:** Read each code snippet once, then rewrite it from memory without looking. Explain
> the concepts out loud if you can't explain it fluently, you're not ready yet.

## 1.1 Object-Oriented Programming (OOP)

OOP rests on four pillars. A surface-level understanding isn't enough you need to know what real-world
problem each one solves.

| Pillar | Meaning | Problem it solves |
|---|---|---|
| **Encapsulation** | Hiding internal state behind an interface | Prevents unwanted manipulation, reduces coupling |
| **Abstraction** | Exposing a contract without implementation details | Lets you swap implementations without changing consumers |
| **Inheritance** | Inheriting behavior from a base class | Reuse, but with the risk of tight coupling |
| **Polymorphism** | Different behavior behind the same interface | Generic code that works across many types |

### Encapsulation and `property`

Python has no true "private" the convention is that a leading underscore means *"internal, don't
touch."* For access control and validation on assignment, we use `property`.

In [ ]:
class Account:
    def __init__(self, balance: float):
        self._balance = balance  # convention: internal

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

acc = Account(100)
acc.balance = 50   # setter runs and validates
# acc.balance = -10  # ValueError


**Interview tip:** The advantage of `property` over explicit getter/setter methods: the interface
stays clean (`acc.balance` instead of `acc.get_balance()`), and you can add validation logic later
without breaking consumer code.

### Abstraction: ABC vs. Protocol

There are two ways to define a contract — know the difference for interviews:

- **ABC (Abstract Base Class)** — explicit inheritance (nominal typing); a class must deliberately
  inherit from it. If an abstract method isn't implemented, you get an error at instantiation.
- **Protocol** — structural typing (duck typing); any class that has the required methods conforms,
  with no inheritance needed. Gives more flexibility for dependency inversion.

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol

# ABC: explicit inheritance required
class Storage(ABC):
    @abstractmethod
    def save(self, key: str, data: bytes) -> None: ...

# Protocol: structural conformance, no inheritance
class Cache(Protocol):
    def get(self, key: str) -> bytes | None: ...
    def set(self, key: str, value: bytes) -> None: ...

class RedisCache:  # doesn't inherit, but conforms
    def get(self, key): ...
    def set(self, key, value): ...

def use(c: Cache): ...  # accepts RedisCache


**Interview tip:** Rule of thumb — use ABC when you control every implementation yourself; use
Protocol for defining dependencies across module boundaries (dependency inversion) and library code,
where it's cleaner and less coupled.

### `dataclass` and `__slots__`

`dataclass` removes boilerplate code (constructor, `__repr__`, `__eq__`) — ideal for simple data models.
`__slots__` reduces memory usage and slightly speeds up attribute access because no `__dict__` is
created — important when you're instantiating millions of objects.

In [ ]:
from dataclasses import dataclass, field

@dataclass(slots=True, frozen=True)  # frozen = immutable and hashable
class Transaction:
    id: str
    amount: float
    tags: list[str] = field(default_factory=list)  # NOT a mutable default value!

t = Transaction("tx1", 42.0)
# t.amount = 10  # FrozenInstanceError because frozen=True


**Note:** Why `field(default_factory=list)`? Because the classic mutable-default-argument trap
applies here too — if you wrote `tags: list = []`, every instance would share the same list.

### Composition over Inheritance

Inheritance creates tight coupling and leads to the "diamond problem" and fragile hierarchies.
Composition (holding an object instead of inheriting from it) is more flexible and more testable.
This is one of the most frequently asked design questions.

In [ ]:
# Instead of inheritance: FraudModel(BaseModel, Logger, Cache) ...

# Composition:
class FraudDetector:
    def __init__(self, model, logger, cache):
        self._model = model    # injected dependencies
        self._logger = logger
        self._cache = cache

    def detect(self, tx):
        self._logger.info("detecting")
        return self._model.predict(tx)


### Dunder methods, `classmethod`, and `staticmethod`

- **Dunder methods** — like `__eq__`, `__hash__`, `__len__`, `__iter__`, `__call__` integrate an
  object's behavior with operators and built-in functions.
- **`classmethod`** — has access to the class, not the instance; useful for alternative constructors
  (factories) like `from_dict`.
- **`staticmethod`** — depends on neither the class nor the instance; used purely to logically group a
  helper function inside a class.

In [ ]:
class User:
    def __init__(self, name): self.name = name

    @classmethod
    def from_dict(cls, d: dict) -> "User":
        return cls(d["name"])  # alternative constructor

    @staticmethod
    def is_valid_name(name: str) -> bool:
        return bool(name.strip())


### Q1.1 — Why do we prefer composition over inheritance?

Inheritance creates tight coupling between a base class and its children; a change in the base class
can break the children (the "fragile base class" problem). Deep hierarchies also make testing and
comprehension harder. Composition keeps components independent, allows swapping and mocking of
dependencies, and avoids the complexity of multiple inheritance. Rule of thumb: **inheritance for a true
"is-a" relationship, composition for "has-a."**

## 1.2 Type Hints and Typing

In professional teams, type hints are effectively mandatory, for three reasons: (1) `mypy` catches
errors before runtime, (2) they improve readability and self-documentation, (3) better IDE support.
The critical point that's always asked: **type hints are not enforced at runtime** — only static
analysis tools use them. Python remains dynamically typed.

### Basic and composite types

In [ ]:
from typing import Optional, Union, Any
from collections.abc import Sequence, Mapping, Callable, Iterable

def mean(xs: Sequence[float]) -> float:  # any sequence, not just list
    return sum(xs) / len(xs)

# Optional[X] is exactly equivalent to X | None (3.10+ pipe syntax preferred)
def find(uid: str) -> Optional[dict]: ...
def find2(uid: str) -> dict | None: ...

# Callable: function signature
Handler = Callable[[dict], bool]  # takes a dict, returns a bool


**Interview tip:** Why `Sequence` instead of `list` for an input parameter? The principle of "be
liberal in what you accept, strict in what you return." `Sequence` accepts both `list` and `tuple`,
making the function more general. For the return type, return a precise type (`list`).

### Generics and `TypeVar`

For writing type-safe generic code (e.g. a Repository that works with any entity), we use `TypeVar` and
`Generic`.

In [ ]:
from typing import TypeVar, Generic

T = TypeVar("T")

class Repository(Generic[T]):
    def __init__(self):
        self._items: dict[str, T] = {}

    def get(self, id: str) -> T | None:
        return self._items.get(id)

    def add(self, id: str, item: T) -> None:
        self._items[id] = item

user_repo: Repository[User] = Repository()  # mypy tracks the type


### Advanced typing tools

- **`Literal`** — restrict to specific values: `Literal['GET', 'POST']`
- **`TypedDict`** — define a dict's structure with specific keys and types (without building a class)
- **`Final`** — a variable that must not be reassigned (a constant)
- **`@overload`** — define multiple signatures for one function
- **`NewType`** — create a distinct type from a base type to prevent mixing (e.g. `UserId` vs. raw `str`)

In [ ]:
from typing import Literal, TypedDict, Final, NewType

class PredictRequest(TypedDict):
    user_id: str
    amount: float
    method: Literal["card", "transfer"]

MAX_RETRIES: Final = 3

UserId = NewType("UserId", str)  # UserId won't be confused with plain str

def load(uid: UserId) -> None: ...
# load("raw")  # mypy warning: not a UserId


### `mypy` in practice

`mypy` analyzes code without executing it. It runs in CI to catch type errors before merge. Strict
mode is the most rigorous option and is recommended for serious projects.

```bash
mypy --strict app/   # every function must have complete annotations
```

### Q1.2 — Does a type hint affect execution speed?

No. Type hints are ignored at runtime (except by tools like Pydantic that deliberately use them at
runtime). They exist purely for static analysis and readability. The only negligible cost is storing
the annotations in `__annotations__`, which is safely ignorable.

## 1.3 Generators and Lazy Evaluation

A generator is a function that produces values one at a time, lazily, instead of building the entire
result in memory. The `yield` keyword turns a function into a generator. This is essential for
processing large data (millions of transactions, multi-gigabyte files) because it brings memory usage
from O(n) down to O(1).

In [ ]:
def read_transactions(path: str):
    with open(path) as f:
        for line in f:              # only one line in memory at a time
            yield parse(line)

# Constant memory usage, regardless of file size
total = sum(t.amount for t in read_transactions("huge.csv"))


### The mechanics of `yield`: suspended state

Every time execution reaches `yield`, it "suspends" and returns a value; on the next call it resumes
from that exact point. This means a generator preserves its internal state between calls.

In [ ]:
def counter():
    n = 0
    while True:      # infinite generator — no memory risk
        yield n
        n += 1

g = counter()
next(g)  # 0
next(g)  # 1 — continued from previous state


### Generator expression vs. list comprehension

| Feature | List comprehension `[...]` | Generator expression `(...)` |
|---|---|---|
| Memory | Entire result built, O(n) | Lazy, O(1) |
| Timing | All computed at once | Computed per `next()` call |
| Multiple iterations | Yes | No (single-use) |
| Use case | When you need the whole list | When you only iterate once |

**Note:** Classic trap — a generator is single-use. If you loop over it again, it's empty because it's
exhausted. If you need to iterate multiple times, convert it to a list or rebuild the generator.

### `yield from` and `itertools`

`yield from` lets a generator delegate to another generator. `itertools` provides powerful lazy tools
for chaining, grouping, and slicing that are heavily used in production.

In [ ]:
import itertools

def flatten(nested):
    for group in nested:
        yield from group   # delegate to each subgroup

# Chunked (batched) processing without loading all the data
def batched(iterable, size):
    it = iter(iterable)
    while batch := list(itertools.islice(it, size)):
        yield batch


### Q1.3 — How do you process a 10 GB file without filling up memory?

Using generators and line-by-line (streaming) processing. I read the file with a generator that yields
one line at a time, and chain the processing as a pipeline of generators. If the output is also large,
I write it in a streaming fashion too. If aggregation is needed (e.g. a sum), I keep it incremental.
This way memory usage stays constant regardless of file size.

## 1.4 Decorators

A decorator is a function that takes a function (or class) and returns an extended version of it,
without changing the original body. This is the "cross-cutting concerns" pattern: logic like logging,
timing, retry, caching, and auth that repeats across many places gets defined once and applied
everywhere.

### Basic structure and the importance of `functools.wraps`

In [ ]:
import functools, time

def timed(fn):
    @functools.wraps(fn)  # preserves the original __name__ and __doc__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            print(f"{fn.__name__}: {time.perf_counter()-start:.3f}s")
    return wrapper

@timed
def predict(x):
    """docstring is preserved"""
    return heavy(x)


**Interview tip:** Why `functools.wraps`? Without it, `__name__` becomes `'wrapper'` and the
docstring is lost. This breaks debugging tools, documentation, and even some frameworks (e.g. FastAPI,
which inspects the signature). This is asked almost every time.

### Decorators with arguments

To pass a parameter to a decorator, we add an extra layer of function (a decorator factory).

In [ ]:
def retry(times=3, delay=0.5):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(times):
                try:
                    return fn(*args, **kwargs)
                except Exception:
                    if attempt == times - 1:
                        raise
                    time.sleep(delay * (2 ** attempt))  # exponential backoff
        return wrapper
    return decorator

@retry(times=5, delay=0.2)
def call_external_api(): ...


### `lru_cache` and real-world uses

`functools.lru_cache` is a ready-made memoization decorator; it caches results of previous calls. Great
for pure, frequently-called functions.

In [ ]:
@functools.lru_cache(maxsize=1024)
def get_country_risk(country: str) -> float:
    return expensive_lookup(country)  # computed only on the first call


**Note:** The `lru_cache` trap — it only works for pure functions with hashable arguments. If the
function has side effects or the underlying data changes, the cache returns stale results.
`maxsize=None` can also lead to a memory leak.

### Decorator stacking order

Multiple decorators apply bottom-up (closest to the function first). Order matters.

In [ ]:
@app.post("/x")     # 3: applied last
@retry(times=3)      # 2
@timed                # 1: first, closest to the function
def handler(): ...

# Equivalent to: app.post(retry(timed(handler)))


### Q1.4 — Write a decorator that measures the execution time of an async function.

For async, the wrapper must also be async and use `await`:

In [ ]:
import functools, time

def timed_async(fn):
    @functools.wraps(fn)
    async def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return await fn(*args, **kwargs)
        finally:
            print(f"{fn.__name__}: {time.perf_counter()-start:.3f}s")
    return wrapper


What the interviewer is looking for: a regular decorator doesn't work for an async function
because it doesn't `await` the coroutine — it just returns the coroutine object without executing it.

## 1.5 Context Managers

A context manager guarantees that resources (files, connections, locks, transactions) are released or
cleaned up even when an error occurs. It works through the `__enter__`/`__exit__` protocol. The `with`
statement guarantees `__exit__` always runs — like `finally`, but cleaner and reusable.

### Class-based implementation

In [ ]:
class DBTransaction:
    def __init__(self, conn):
        self.conn = conn

    def __enter__(self):
        self.tx = self.conn.begin()
        return self.tx

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.tx.commit()
        else:
            self.tx.rollback()  # roll back on error
        return False  # False = re-raise the exception


**Note:** The return value of `__exit__` matters — if you return `True`, the exception is
swallowed (suppressed). You usually want `False` (or `None`) so the error propagates upward.
Accidentally returning `True` is a common silent bug.

### The easier way: `@contextmanager`

In [ ]:
from contextlib import contextmanager

@contextmanager
def db_transaction(conn):
    tx = conn.begin()
    try:
        yield tx     # code inside the `with` block runs here
        tx.commit()
    except Exception:
        tx.rollback()
        raise
    finally:
        tx.close()

with db_transaction(conn) as tx:
    tx.execute("UPDATE ...")


### `ExitStack` and async context managers

`ExitStack` manages a variable number of context managers (e.g. opening N files). For async resources
(async connections, HTTP sessions) we use `async with` and `__aenter__`/`__aexit__`.

In [ ]:
from contextlib import ExitStack, asynccontextmanager

# Multiple dynamic resources
with ExitStack() as stack:
    files = [stack.enter_context(open(p)) for p in paths]
    # all closed at the end

@asynccontextmanager
async def get_session():
    session = await create_session()
    try:
        yield session
    finally:
        await session.close()

async with get_session() as s: ...


### Q1.5 — What's the difference between a context manager and try/finally?

Both guarantee cleanup, but a context manager encapsulates the setup and teardown logic into a reusable
unit. Instead of repeating `try/finally` everywhere it's used, you write the logic once in
`__enter__`/`__exit__` and reuse it everywhere via `with`. It's more readable and less prone to
forgetting cleanup. Under the hood, `with` uses the same `try/finally` mechanism.

## 1.6 Async Programming

`async`/`await` multiplies throughput for I/O-bound work (network, database, queues, files). Core idea:
while waiting for an I/O response, instead of blocking the thread, control returns to the event loop so
it can do other work. This is **concurrency**, not necessarily **parallelism**.

### Concurrency vs. parallelism

| Concept | What it is | Python tool | Best for |
|---|---|---|---|
| Concurrency | Managing multiple tasks by rapid switching | `asyncio` | I/O-bound |
| Parallelism | Truly simultaneous execution across cores | `multiprocessing` | CPU-bound |
| Threading | Multiple threads, but limited by the GIL | `threading` | Simple I/O-bound |

### Basic pattern and `gather`

In [ ]:
import asyncio, httpx

async def fetch(client, uid):
    r = await client.get(f"/score/{uid}")
    return r.json()

async def main(uids):
    async with httpx.AsyncClient() as client:
        # all requests concurrent, not sequential
        return await asyncio.gather(*[fetch(client, u) for u in uids])

# Running 100 requests: instead of the sum of all latencies, roughly the slowest one


### Limiting concurrency with `Semaphore`

Gathering 10,000 tasks at once can overwhelm the target server. A `Semaphore` caps concurrency.

In [ ]:
sem = asyncio.Semaphore(20)  # max 20 concurrent

async def fetch_limited(client, uid):
    async with sem:
        return await fetch(client, uid)


### Common async pitfalls

- **Blocking the event loop:** calling slow sync code (e.g. `time.sleep`, a sync DB query, or heavy
  computation) freezes the entire loop. Fix: use the async version, or `run_in_executor`.
- **Forgetting `await`:** calling a coroutine without `await` just creates an object without executing
  it (a silent bug).
- **Fire-and-forget without keeping a reference:** a task whose reference isn't kept may get
  garbage-collected.

In [ ]:
# Wrong: blocks the loop
async def bad():
    time.sleep(5)   # freezes the entire server!

# Correct:
async def good():
    await asyncio.sleep(5)   # lets the loop keep working

# Running CPU-bound code without blocking the loop
loop = asyncio.get_event_loop()
result = await loop.run_in_executor(None, heavy_cpu_function, data)


### Q1.6 — Why don't we use `asyncio` for CPU-bound work?

Because `asyncio` is single-threaded, and its concurrency is based on "yielding control while waiting
for I/O." CPU-bound work has no waiting point — it keeps the processor occupied continuously and never
returns control to the event loop, so other tasks starve. On top of that, the GIL prevents true thread
parallelism. The right approach for CPU-bound work is `multiprocessing` (or running that work in a
process pool via `run_in_executor`).

## 1.7 Time and Space Complexity

Big-O notation describes how resource usage (time or memory) grows relative to input size, ignoring
constants. In an interview you must state the complexity of your solution and know which data structure
improves it. This is the difference between an accepted solution and a rejected, slow one.

| Big-O | Name | Example |
|---|---|---|
| O(1) | Constant | `dict`/`set` access, list indexing |
| O(log n) | Logarithmic | Binary search, balanced tree |
| O(n) | Linear | Iterating a list |
| O(n log n) | Linearithmic | Efficient sorting (`sorted`) |
| O(n²) | Quadratic | Two nested loops |
| O(2ⁿ) | Exponential | Recursion without memoization |

### Complexity of core data structures

| Operation | list | dict / set | deque |
|---|---|---|---|
| Access by index/key | O(1) | O(1) average | O(n) |
| Search for a value (`in`) | O(n) | O(1) average | O(n) |
| Append at end | O(1)* | O(1) | O(1) |
| Insert/remove at start | O(n) | — | O(1) |

**Interview tip:** Why is `set`/`dict` membership O(1)? Because they're hash tables: the key is hashed
and goes directly to a bucket. A `list` must scan linearly. If you're repeatedly checking
`x in collection` and the collection is large, convert it to a `set` — this is the single most common
optimization in coding interviews.

### Space complexity and amortized cost

Space complexity is the extra memory consumed relative to the input. Sometimes we trade time for
memory (e.g. using a `set` to get O(1) lookup at the cost of O(n) memory). "Amortized O(1)" means that
even though one operation is occasionally expensive (e.g. resizing the array backing a list), the
average over time is O(1).

### Q1.7 — What is the complexity of the following code, and how would you improve it?

Suppose we're looking for elements common to two large lists:

In [ ]:
# Slow version: O(n*m)
common = [x for x in list_a if x in list_b]

# Optimized version: O(n+m)
set_b = set(list_b)                         # O(m), once
common = [x for x in list_a if x in set_b]  # each lookup is now O(1)


The first version scans the entire `list_b` for every element of `list_a` (O(n·m)). By converting
`list_b` to a `set`, lookups become O(1) and the whole algorithm drops to O(n+m). This is exactly the
kind of analysis the interviewer wants to hear.

## 1.8 Clean Code and Engineering Principles

Clean code is code that a human can easily read, change, and test. A tech lead cares about this because
code spends most of its life being *maintained*, not being written for the first time. Key principles:

### SOLID principles (brief)

- **S — Single Responsibility:** every class/function should have one reason to change.
- **O — Open/Closed:** open for extension, closed for modification (via abstraction, not long
  if/elif chains).
- **L — Liskov Substitution:** a subclass must be substitutable for its base class without breaking
  behavior.
- **I — Interface Segregation:** small, focused interfaces are better than one large interface.
- **D — Dependency Inversion:** depend on abstractions, not concrete implementations (the same idea as
  Protocol / dependency injection).

### Clean-code habits

- Short, single-responsibility functions; if a function's description needs "and," it probably should
  be split.
- Meaningful naming; good code needs fewer comments. A comment should explain *why*, not *what*.
- Separate business logic from I/O; test the pure logic, mock the I/O.
- Avoid mutable default arguments, magic numbers, and functions with too many parameters (more than
  3–4 → use a `dataclass`).
- Fail fast: reject invalid input early (guard clauses) instead of nesting conditionals.

### The classic trap: mutable default argument

In [ ]:
# Trap:
def add_item(item, items=[]):   # the list is created only once!
    items.append(item)
    return items

add_item(1)  # [1]
add_item(2)  # [1, 2]  <- shared list! unexpected result

# Correct:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


### Guard clauses vs. deep nesting

In [ ]:
# Hard to read, deeply nested:
def process(user):
    if user is not None:
        if user.active:
            if user.balance > 0:
                return do_work(user)

# With guard clauses — flat and readable:
def process(user):
    if user is None:
        return None
    if not user.active:
        return None
    if user.balance <= 0:
        return None
    return do_work(user)


### Q1.8 — How do you write code that's testable?

A few principles: (1) I separate business logic from I/O so I can test pure logic without a
database/network. (2) I inject dependencies so I can substitute mocks in tests. (3) I keep functions
small and single-responsibility so test units stay small. (4) I avoid global state and hidden side
effects, since they make tests non-deterministic. (5) I keep function outputs deterministic; I make
non-deterministic sources like time and randomness injectable.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **OOP:** Encapsulation via `property`; abstraction via ABC (explicit inheritance) or Protocol
  (structural); composition > inheritance; `dataclass` for data models.
- **Typing:** Not enforced at runtime; `Optional[X]` = `X | None`; `Sequence` on input, precise type on
  output; Generics with `TypeVar`; `mypy` in CI.
- **Generators:** `yield` for laziness and O(1) memory; single-use; streaming for large data.
- **Decorators:** Always use `functools.wraps`; a factory for parameters; `lru_cache` only for pure
  functions; the wrapper must be async too for async functions.
- **Context managers:** Guarantee cleanup; `@contextmanager` is simpler; an `__exit__` that returns
  `True` swallows the exception.
- **Async:** For I/O-bound work; never block the loop; `gather` for concurrency; `Semaphore` to limit
  it; CPU-bound → `multiprocessing`.
- **Complexity:** `set`/`dict` for O(1) membership; avoid O(n²); trade time for memory when useful.
- **Clean code:** SOLID; small functions; guard clauses; separate logic from I/O for testability; avoid
  mutable defaults.